# YeastCaduceus — Phase 4: Variant Effect Prediction & Benchmarks

Two VEP scorings (both pinned in `dir/YeastCaduceus_WBS.xlsx`):

1. **Zero-shot** (MLM adapter alone, *no supervised head*) — mask the variant
   position (centered at the 4096th token, per PlantCAD2's conservation method),
   score `log p(ref) − log p(alt)` from the lm_head distribution. Usable the moment
   Phase 2 finishes.
2. **Supervised** (needs Phase 3 head) — `log₂(Cov_alt+1) − log₂(Cov_ref+1)` summed
   over the exon bins overlapping the variant's gene.

Benchmarks (AUROC/AUPRC vs Shorkie 0.85–0.87): **Caudal** cis-eQTL (~1,901),
**Kita** eQTL (683, stratified by location), **DREAM** MPRA (71,103, Spearman).

> ⚠️ Scaffold. The scoring functions are the load-bearing, reusable part. Dataset
> loaders are **stubs** — fill the real URLs/parsers (Shorkie's `data/` + the paper
> supplements) and the R64 exon annotation before running.


In [ ]:
# CELL 1 — Install  (identical 4-step order — see notebooks 00/02/03)
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121 -q
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 -q
!pip3 install causal-conv1d==1.4.0 -q
!pip3 install transformers==4.46.3 peft==0.14.0 -q
!pip3 install biopython pyfaidx pyBigWig datasets scikit-learn scipy pandas numpy tqdm huggingface_hub -q
print('✅ Cell 1 — install done (verify versions as in notebook 00 Cell 1)')

In [ ]:
# CELL 2 — Paths, model id, benchmark dataset locations
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
BASE          = Path('/content/drive/MyDrive/yeastcaduceus')
MLM_ADAPTER   = BASE / 'checkpoints/phase2/final_adapter'          # TODO verify
SUP_ADAPTER   = BASE / 'checkpoints/phase3/final_adapter'          # TODO verify
SUP_HEAD      = BASE / 'checkpoints/phase3/coverage_head.pt'
R64_FASTA     = Path('/content/drive/MyDrive/shorkie/data/backup/genomes/R64')  # dir; pick .fasta
BENCH_DIR     = BASE / 'data/benchmarks'
RESULTS_DIR   = BASE / 'results/phase4'
MODEL_ID      = 'kuleshov-group/PlantCAD2-Small-l24-d0768'
WINDOW_SIZE, CENTER = 8192, 4096        # variant placed at the centre token
for d in (BENCH_DIR, RESULTS_DIR): d.mkdir(parents=True, exist_ok=True)

# Benchmark datasets — Shorkie reused these; fill real sources before running.
BENCHMARKS = {
    'caudal_eqtl': {'n': 1901, 'src': 'TODO: Caudal et al. cis-eQTL table (Shorkie data/ or paper supp)'},
    'kita_eqtl':   {'n': 683,  'src': 'TODO: Kita et al. eQTL (Promoter/UTR5/UTR3/ORF strata)'},
    'dream_mpra':  {'n': 71103,'src': 'TODO: DREAM Challenge MPRA synthetic promoters (Rafi et al.)'},
}
print('✅ Cell 2 done — benchmarks:', list(BENCHMARKS))

In [ ]:
# CELL 3 — Load tokenizer + MLM-adapted backbone (zero-shot path)
# Loads base → merges Phase-2 MLM LoRA → exposes lm_head for masked scoring.
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
_nuc = {n: tokenizer(n, add_special_tokens=False)['input_ids'][0] for n in 'ACGT'}
NUC_ID = _nuc                       # {'A':3,'C':4,'G':5,'T':6}
MASK_ID = tokenizer.mask_token_id

mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True,
                                           torch_dtype=torch.bfloat16)
if MLM_ADAPTER.exists():
    mlm = PeftModel.from_pretrained(mlm, str(MLM_ADAPTER)).merge_and_unload()
    print('  ✅ Phase-2 MLM adapter merged')
else:
    print('  ⚠️ No MLM adapter — scoring with the BASE backbone (sanity only)')
mlm = mlm.eval().to('cuda:0')
print('✅ Cell 3 done — nuc ids', NUC_ID, 'mask', MASK_ID)

In [ ]:
# CELL 4 — Zero-shot VEP:  log p(ref) − log p(alt)   (masked, variant-centred)
# seq: 8192bp string (ref allele at index CENTER). ref/alt: single bases.
import torch, torch.nn.functional as F

@torch.no_grad()
def zeroshot_vep(seq, ref, alt, pos=CENTER):
    assert len(seq) == WINDOW_SIZE and seq[pos].upper() == ref.upper(), 'ref must sit at pos'
    ids = tokenizer(seq.upper(), return_tensors='pt')['input_ids'].to('cuda:0')
    ids[0, pos] = MASK_ID                                  # mask the variant site
    logits = mlm(input_ids=ids).logits[0, pos]             # [VOCAB]
    logp = F.log_softmax(logits.float(), dim=-1)
    return (logp[NUC_ID[ref.upper()]] - logp[NUC_ID[alt.upper()]]).item()

# sanity: ref==alt → 0
_s = 'ACGT'*2048
print('zero-shot self-score (expect ~0):', round(zeroshot_vep(_s, _s[CENTER], _s[CENTER]), 4))
print('✅ Cell 4 done')

In [ ]:
# CELL 5 — Supervised VEP:  log2(Cov_alt+1) − log2(Cov_ref+1) over exon bins
# Needs the Phase-3 CoverageModel (import its definition or re-run notebook 03
# Cells 3+5 to build `model`). exon_bins = boolean mask over the 512 bins that
# overlap the variant's gene exons (from R64 annotation — TODO loader).
import numpy as np, torch, torch.nn.functional as F

@torch.no_grad()
def supervised_vep(model, seq_ref, seq_alt, exon_bins=None, tracks=None):
    def cov(s):
        ids = tokenizer(s.upper(), return_tensors='pt')['input_ids'].to('cuda:0')
        return model(input_ids=ids)['pred'][0].float().cpu().numpy()   # [512, T]
    cr, ca = cov(seq_ref), cov(seq_alt)
    if tracks is not None: cr, ca = cr[:, tracks], ca[:, tracks]
    if exon_bins is not None: cr, ca = cr[exon_bins], ca[exon_bins]
    return float((np.log2(ca.sum()+1) - np.log2(cr.sum()+1)))

print('✅ Cell 5 defined (supervised_vep — requires notebook 03 `model`)')

In [ ]:
# CELL 6 — Benchmark harness: AUROC/AUPRC (eQTL) and Spearman (MPRA)
# Expects a scored dataframe with columns: score, label  (label 1=causal/expressed).
import numpy as np, json
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

def eval_classification(scores, labels):
    s = np.abs(np.asarray(scores, float))     # |effect| for direction-agnostic eQTL
    y = np.asarray(labels, int)
    return {'auroc': float(roc_auc_score(y, s)),
            'auprc': float(average_precision_score(y, s)),
            'n': int(len(y))}

def eval_regression(scores, measured):
    rho = spearmanr(scores, measured).correlation
    return {'spearman': float(rho), 'n': int(len(scores))}

# stratified variant — Kita is reported by genomic location:
def eval_stratified(scores, labels, strata):
    out = {}
    scores, labels, strata = map(np.asarray, (scores, labels, strata))
    for g in np.unique(strata):
        m = strata == g
        if m.sum() > 5 and len(np.unique(labels[m])) == 2:
            out[str(g)] = eval_classification(scores[m], labels[m])
    return out

print('✅ Cell 6 — harness ready (eval_classification / eval_regression / eval_stratified)')

In [ ]:
# CELL 7 — Dataset loaders (STUBS) + run all benchmarks
# Each loader returns a DataFrame with at least: seq(8192bp, ref-centred), ref, alt, label
# (and 'stratum' for Kita, 'measured' for DREAM). Fill from BENCHMARKS['*']['src'].
import pandas as pd, json

def load_caudal():  raise NotImplementedError('TODO: parse Caudal cis-eQTL → seq/ref/alt/label')
def load_kita():    raise NotImplementedError('TODO: parse Kita eQTL → +stratum (Promoter/UTR5/UTR3/ORF)')
def load_dream():   raise NotImplementedError('TODO: parse DREAM MPRA → seq/measured')

def run_caudal():
    df = load_caudal()
    df['score'] = [zeroshot_vep(r.seq, r.ref, r.alt) for r in df.itertuples()]
    return eval_classification(df['score'], df['label'])

def run_kita():
    df = load_kita()
    df['score'] = [zeroshot_vep(r.seq, r.ref, r.alt) for r in df.itertuples()]
    return {'overall': eval_classification(df['score'], df['label']),
            'by_location': eval_stratified(df['score'], df['label'], df['stratum'])}

# Example (uncomment once loaders are implemented):
# results = {'caudal': run_caudal(), 'kita': run_kita()}
# (RESULTS_DIR/'phase4_metrics.json').write_text(json.dumps(results, indent=2))
# print(results)
print('✅ Cell 7 — wire up loaders, then run_caudal()/run_kita(). Compare vs Shorkie 0.85–0.87.')